In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from IPython.display import clear_output
from numpy import arange
import os 
from sklearn.metrics import r2_score
import ta
from ta import *
import talib
#from rfpimp import permutation_importances


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import ta
import warnings
import time 
from IPython.display import clear_output
from pyti.exponential_moving_average import exponential_moving_average as EMA
import xlrd 
import xlsxwriter
import openpyxl
import sys
import decimal
import pyodbc 
import pandas.io.sql
import smtplib
import mimetypes
from email.mime.multipart import MIMEMultipart
from email import encoders
from email.message import Message
from email.mime.audio import MIMEAudio
from email.mime.base import MIMEBase
from email.mime.image import MIMEImage
from email.mime.text import MIMEText
from ta import *
import talib
import time
import datetime
import os
#import paho.mqtt.client as mqtt
import json
import mysql.connector
from mysql.connector import errorcode
from elasticsearch import Elasticsearch
import socket 
import sys  
import csv
import socket
from datetime import datetime
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
from pyspark.sql import SQLContext
from collections import namedtuple
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_prominences
import pickle
import matplotlib
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib import style
from functools import reduce 
import re
import seaborn as sns
%matplotlib inline
import sklearn
from sklearn import preprocessing
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import glob


# Section  1.   Variable Creation

In [ ]:
# ####### importing the raw data for H4 AND D1 tinme frame 

#path = "<local-path>"

# path = "<local-path>"


# df_h1= pd.read_csv(path + "H1 Raw Data - 10 years.csv")
# df_h4= pd.read_csv(path + "H4 Raw Data - 10 years.csv")
# df_d1= pd.read_csv(path + "D1 Raw Data - 10 years.csv")


#path_sep_oct_data = "<local-path>"
#df_h1= pd.read_csv(path_sep_oct_data + "H1 BROKER 4 month data for TP optimization.csv")
#df_h4= pd.read_csv(path_sep_oct_data + "H4 BROKER 4 month data for TP optimization.csv")
#df_d1= pd.read_csv(path_sep_oct_data + "D1 BROKER 4 month data for TP optimization.csv")





path_sep_oct_data = "<local-path>"



df_h1= pd.read_csv(path_sep_oct_data + "H1 BROKER data.csv")
df_h4= pd.read_csv(path_sep_oct_data + "H4 BROKER data.csv")
df_d1= pd.read_csv(path_sep_oct_data + "D1 BROKER data.csv")









df_h1 = df_h1.rename(columns = {'H1_avg_open':'askopen','H1_avg_high':'askhigh',
                              'H1_avg_low':'asklow','H1_avg_close':'askclose',
                              'H1_volume':'_volume','H1_volume_new':'_volume_new',
                             'H1_Timeframe':'TimeFrame'})

#df_h4 = df_h4.drop(columns = ['Unnamed: 0'])

df_h4 = df_h4.rename(columns = {'H4_avg_open':'askopen','H4_avg_high':'askhigh',
                              'H4_avg_low':'asklow','H4_avg_close':'askclose',
                              'H4_volume':'_volume','H4_volume_new':'_volume_new',
                             'H4_Timeframe':'TimeFrame'})

df_d1 = df_d1.rename(columns = {'D1_avg_open':'askopen','D1_avg_high':'askhigh',
                              'D1_avg_low':'asklow','D1_avg_close':'askclose',
                              'D1_volume':'_volume','D1_volume_new':'_volume_new',
                              'D1_Timeframe':'TimeFrame'})





df = df_h1.append(df_h4)
df = df.append(df_d1)
# df = df.rename(columns = {'Timeframe':'TimeFrame','tickqty':'_volume'})
df.head()



# #path = "<local-path>"

# path = "<local-path>"


# # df_h1= pd.read_csv(path + "D1 data 2020 latest.csv")
# df_h4= pd.read_csv(path + "H4 Raw Data - 10 years.csv")
# df = df_h4.head(500)
# # df_d1= pd.read_csv(path + "D1 Raw Data - 10 years.csv")



# # df = df_h1.append(df_h4)
# # df = df.append(df_d1)
# df = df.rename(columns = {'Timeframe':'TimeFrame','tickqty':'_volume'})
# df.head()



In [ ]:
len(df['Currency'].unique())

In [ ]:
df['Currency'].unique()

In [ ]:
#### getting the count of rows for each timeframe by currency 
row_count = pd.DataFrame()
row_count = pd.DataFrame(df.groupby(by=['Currency','TimeFrame'])['date'].count())
row_count.sort_values(by=['Currency'])

In [ ]:
##### Code for creation of variables for the models 

In [ ]:

####


def model_data_prep(Currency,Timeframe):
    
    global final_data
    global counter
    counter+=1
    
    print("Currency-----> "  + str(Currency))
    print("Timeframe-----> "  + str(Timeframe))
    print("Counter-----> "  + str(counter))
    


    df_prep =df[(df['TimeFrame'] == Timeframe) & (df['Currency'] == Currency)]
    df_prep = df_prep.reset_index(drop = True)


    #-------------Making of Stochastic And EMA Data---------------------------#


    df_prep['_stock_%K'] = ta.momentum.stoch(df_prep['askhigh'],df_prep['asklow'],\
                                                         df_prep['askclose'], n=20, fillna=False)

    df_prep['_stoch_%D'] = ta.momentum.stoch_signal(df_prep['askhigh'],df_prep['asklow'],\
                                                                df_prep['askclose'], n=20, d_n=6, fillna=False)

    df_prep['_stoch_slow_%K'] = df_prep['_stoch_%D']

    df_prep['_stoch_slow_%D'] = df_prep['_stoch_slow_%K'].rolling(window=6).mean()

    df_prep['_stoch_direction'] = np.where((df_prep['_stoch_slow_%K'])>(df_prep['_stoch_slow_%D']),1,0).astype('float32')
    df_prep['_stoch_slow_0_1'] = df_prep['_stoch_direction']



    df_prep['_Ema_20'] = ta.trend.ema_indicator(df_prep['askclose'], n=20, fillna=False)

    df_prep['_Ema_200'] = ta.trend.ema_indicator(df_prep['askclose'], n=200, fillna=False)

    df_prep['Ema_direction'] = np.where((df_prep['_Ema_20'])>(df_prep['_Ema_200']),1,0).astype('float32')

    #df_prep['Ema_0_1'] = df_prep['_Ema_direction']

    df_prep = add_all_ta_features(df_prep, "askopen","askhigh","asklow",\
                                     "askclose","_volume", fillna=False)


    #-------------Making of CandleStick Data ------------------------------------#

    op = df_prep['askopen']
    hi = df_prep['askhigh']
    lo = df_prep['asklow']
    cl = df_prep['askclose']

    #------------DOJI---------------#

    df_prep['CDLDOJI']            = talib.CDLDOJI(op, hi, lo, cl)
    df_prep['CDLDOJISTAR']        = talib.CDLDOJISTAR(op, hi, lo, cl)
    df_prep['CDLMORNINGDOJISTAR'] = talib.CDLMORNINGDOJISTAR(op, hi, lo, cl)
    df_prep['CDLEVENINGDOJISTAR'] = talib.CDLEVENINGDOJISTAR(op, hi, lo, cl)
    df_prep['CDLDRAGONFLYDOJI']   = talib.CDLDRAGONFLYDOJI(op, hi, lo, cl)
    df_prep['CDLGRAVESTONEDOJI']  = talib.CDLGRAVESTONEDOJI(op, hi, lo, cl)

        #------------STAR---------------#

    df_prep['CDLMORNINGSTAR']  = talib.CDLMORNINGSTAR(op, hi, lo, cl)
    df_prep['CDLSHOOTINGSTAR'] = talib.CDLSHOOTINGSTAR(op, hi, lo, cl)
    df_prep['CDLEVENINGSTAR']  = talib.CDLEVENINGSTAR(op, hi, lo, cl)


        #------------HAMMER--------------#

    df_prep['CDLHAMMER']         = talib.CDLHAMMER(op, hi, lo, cl)
    df_prep['CDLHANGINGMAN']     = talib.CDLHANGINGMAN(op, hi, lo, cl)
    df_prep['CDLINVERTEDHAMMER'] = talib.CDLINVERTEDHAMMER(op, hi, lo, cl)


        #------------HARAMI---------------#

    df_prep['CDLHARAMI']      = talib.CDLHARAMI(op, hi, lo, cl)
    df_prep['CDLHARAMICROSS'] = talib.CDLHARAMICROSS(op, hi, lo, cl)

       #-----------ALL PATTERNS---------#

    df_prep['CDL2CROWS']           = talib.CDL2CROWS (op, hi, lo, cl)
    df_prep['CDL3BLACKCROWS']      = talib.CDL3BLACKCROWS(op, hi, lo, cl)
    df_prep['CDL3INSIDE']          = talib.CDL3INSIDE(op, hi, lo, cl)
    df_prep['CDL3LINESTRIKE']      = talib.CDL3LINESTRIKE (op, hi, lo, cl)
    df_prep['CDL3OUTSIDE']         = talib.CDL3OUTSIDE(op, hi, lo, cl)
    df_prep['CDL3STARSINSOUTH']    = talib.CDL3STARSINSOUTH(op, hi, lo, cl)
    df_prep['CDL3WHITESOLDIERS']   = talib.CDL3WHITESOLDIERS  (op, hi, lo, cl)
    df_prep['CDLABANDONEDBABY']    = talib.CDLABANDONEDBABY(op, hi, lo, cl)
    df_prep['CDLADVANCEBLOCK']     = talib.CDLADVANCEBLOCK (op, hi, lo, cl)
    df_prep['CDLBELTHOLD']         = talib.CDLBELTHOLD(op, hi, lo, cl)
    df_prep['CDLBREAKAWAY']        = talib.CDLBREAKAWAY  (op, hi, lo, cl)
    df_prep['CDLCLOSINGMARUBOZU']  = talib.CDLCLOSINGMARUBOZU(op, hi, lo, cl)
    df_prep['CDLCONCEALBABYSWALL'] = talib.CDLCONCEALBABYSWALL  (op, hi, lo, cl)
    df_prep['CDLCOUNTERATTACK']    = talib.CDLCOUNTERATTACK(op, hi, lo, cl)
    df_prep['CDLDARKCLOUDCOVER']   = talib.CDLDARKCLOUDCOVER (op, hi, lo, cl)
    df_prep['CDLGAPSIDESIDEWHITE'] = talib.CDLGAPSIDESIDEWHITE  (op, hi, lo, cl)
    df_prep['CDLHIGHWAVE']         = talib.CDLHIGHWAVE (op, hi, lo, cl)
    df_prep['CDLHIKKAKE']          = talib.CDLHIKKAKE(op, hi, lo, cl)
    df_prep['CDLHIKKAKEMOD']       = talib.CDLHIKKAKEMOD(op, hi, lo, cl)
    df_prep['CDLHOMINGPIGEON']     = talib.CDLHOMINGPIGEON (op, hi, lo, cl)
    df_prep['CDLIDENTICAL3CROWS']  = talib. CDLIDENTICAL3CROWS (op, hi, lo, cl)
    df_prep['CDLINNECK']           = talib.CDLINNECK(op, hi, lo, cl)
    df_prep['CDLINVERTEDHAMMER']   = talib.CDLINVERTEDHAMMER (op, hi, lo, cl)
    df_prep['CDLKICKING']          = talib.CDLKICKING(op, hi, lo, cl)
    df_prep['CDLKICKINGBYLENGTH']  = talib.CDLKICKINGBYLENGTH(op, hi, lo, cl)
    df_prep['CDLLADDERBOTTOM']     = talib.CDLLADDERBOTTOM (op, hi, lo, cl)
    df_prep['CDLLONGLEGGEDDOJI']   = talib.CDLLONGLEGGEDDOJI (op, hi, lo, cl)
    df_prep['CDLLONGLINE']         = talib.CDLLONGLINE  (op, hi, lo, cl)
    df_prep['CDLMARUBOZU']         = talib.CDLMARUBOZU(op, hi, lo, cl)
    df_prep['CDLMATCHINGLOW']      = talib.CDLMATCHINGLOW(op, hi, lo, cl)
    df_prep['CDLMATHOLD']          = talib.CDLMATHOLD  (op, hi, lo, cl)
    df_prep['CDLONNECK']           = talib.CDLONNECK (op, hi, lo, cl)
    df_prep['CDLPIERCING']         = talib.CDLPIERCING(op, hi, lo, cl)
    df_prep['CDLRICKSHAWMAN']      = talib.CDLRICKSHAWMAN(op, hi, lo, cl)
    df_prep['CDLRISEFALL3METHODS'] = talib.CDLRISEFALL3METHODS  (op, hi, lo, cl)
    df_prep['CDLSEPARATINGLINES']  = talib.CDLSEPARATINGLINES(op, hi, lo, cl)
    df_prep['CDLSHORTLINE']        = talib.CDLSHORTLINE  (op, hi, lo, cl)
    df_prep['CDLSPINNINGTOP']      = talib.CDLSPINNINGTOP(op, hi, lo, cl)
    df_prep['CDLSTALLEDPATTERN']   = talib.CDLSTALLEDPATTERN (op, hi, lo, cl)
    df_prep['CDLSTICKSANDWICH']    = talib.CDLSTICKSANDWICH (op, hi, lo, cl)
    df_prep['CDLTAKURI']           = talib.CDLTAKURI (op, hi, lo, cl)
    df_prep['CDLTASUKIGAP']        = talib.CDLTASUKIGAP  (op, hi, lo, cl)
    df_prep['CDLTHRUSTING']        = talib.CDLTHRUSTING  (op, hi, lo, cl)
    df_prep['CDLTRISTAR']          = talib.CDLTRISTAR(op, hi, lo, cl)
    df_prep['CDLUNIQUE3RIVER']     = talib.CDLUNIQUE3RIVER (op, hi, lo, cl)
    df_prep['CDLUPSIDEGAP2CROWS']  = talib.CDLUPSIDEGAP2CROWS(op, hi, lo, cl)
    df_prep['CDLXSIDEGAP3METHODS'] = talib.CDLXSIDEGAP3METHODS (op, hi, lo, cl)


    #---------------------------Making Price Action------------------------------------------#
    df_prep['Price_wrt_max_last_50'] = (df_prep['askclose'].rolling(window=50).max()/df_prep['askclose']) - 1
    df_prep['Price_wrt_max_last_100'] = (df_prep['askclose'].rolling(window=100).max()/df_prep['askclose']) - 1
    df_prep['Price_wrt_max_last_150'] = (df_prep['askclose'].rolling(window=150).max()/df_prep['askclose']) - 1
    
    df_prep['Price_wrt_min_last_50'] = 1-(df_prep['askclose'].rolling(window=50).min()/df_prep['askclose']) 
    df_prep['Price_wrt_min_last_100'] = 1-(df_prep['askclose'].rolling(window=100).min()/df_prep['askclose']) 
    df_prep['Price_wrt_min_last_150'] = 1-(df_prep['askclose'].rolling(window=150).min()/df_prep['askclose']) 
    
    
    col_names_candles = df_prep.columns
    varlist_candles = [col for col in col_names_candles if col.startswith('CDL')]

    for item in varlist_candles:
        #---converting the candle value into 0 and 1---#
        df_prep[item] = df_prep[item].astype(bool).astype(int)
        it = item[0:len(item)]
        #-----Making a rolling sum of 5------#
        df_prep[it+"_Rolling_5"] = df_prep[item].rolling(window=5).sum()
        
    
    
    
    ##---- MIN and MAX price---##
    df_prep['Price_max_last_50'] = (df_prep['askclose'].rolling(window=50).max())
    df_prep['Price_max_last_100'] = (df_prep['askclose'].rolling(window=100).max())
    df_prep['Price_max_last_150'] = (df_prep['askclose'].rolling(window=150).max())
    
    df_prep['Price_min_last_50'] = (df_prep['askclose'].rolling(window=50).min())
    df_prep['Price_min_last_100'] = (df_prep['askclose'].rolling(window=100).min())
    df_prep['Price_min_last_150'] = (df_prep['askclose'].rolling(window=150).min())
    
    
    
    ## ------ OverBought and OverSold Duration -----##

    #Stochastic Oscillator
    df_prep['Stoch_overbought'] = np.where(df_prep['_stock_%K']>80,1,0)
    df_prep['Stoch_oversold'] = np.where(df_prep['_stock_%K']<20,1,0)
    
    df_prep['Stoch_overbought_%'] = ((df_prep['Stoch_overbought'].rolling(window=50).sum())/50)*100
    df_prep['Stoch_oversold_%'] = ((df_prep['Stoch_oversold'].rolling(window=50).sum())/50)*100

    
    #Relative Strength Index
    df_prep['RSI_overbought'] = np.where(df_prep['momentum_rsi']>70,1,0)
    df_prep['RSI_oversold'] = np.where(df_prep['momentum_rsi']<30,1,0)
    
    df_prep['RSI_overbought_%'] = ((df_prep['RSI_overbought'].rolling(window=50).sum())/50)*100
    df_prep['RSI_oversold_%'] = ((df_prep['RSI_oversold'].rolling(window=50).sum())/50)*100
    
    #True strength index
    df_prep['TSI_overbought'] = np.where(df_prep['momentum_tsi']>0,1,0)
    df_prep['TSI_oversold'] = np.where(df_prep['momentum_tsi']<0,1,0)
    
    df_prep['TSI_overbought_%'] = ((df_prep['TSI_overbought'].rolling(window=50).sum())/50)*100
    df_prep['TSI_oversold_%'] = ((df_prep['TSI_oversold'].rolling(window=50).sum())/50)*100
    
    
    #Ultimate Oscillator
    df_prep['UO_overbought'] = np.where(df_prep['momentum_uo']>70,1,0)
    df_prep['UO_oversold'] = np.where(df_prep['momentum_uo']<30,1,0)
    
    df_prep['UO_overbought_%'] = ((df_prep['UO_overbought'].rolling(window=50).sum())/50)*100
    df_prep['UO_oversold_%'] = ((df_prep['UO_oversold'].rolling(window=50).sum())/50)*100
    
    
    #Williams %R
    df_prep['Williams %R_overbought'] = np.where(((df_prep['momentum_wr']<0) & (df_prep['momentum_wr']>-20)),1,0)
    df_prep['Williams %R_oversold'] = np.where(((df_prep['momentum_wr']>-80) & (df_prep['momentum_wr']<-100)),1,0)
    
    df_prep['Williams %R_overbought_%'] = ((df_prep['Williams %R_overbought'].rolling(window=50).sum())/50)*100
    df_prep['Williams %R_oversold_%'] = ((df_prep['Williams %R_oversold'].rolling(window=50).sum())/50)*100
    
    
    #Money Flow Index
    df_prep['MFI_overbought'] = np.where(df_prep['momentum_mfi']>80,1,0)
    df_prep['MFI_oversold'] = np.where(df_prep['momentum_mfi']<20,1,0)
    
    df_prep['MFI_overbought_%'] = ((df_prep['MFI_overbought'].rolling(window=50).sum())/50)*100
    df_prep['MFI_oversold_%'] = ((df_prep['MFI_oversold'].rolling(window=50).sum())/50)*100
    
    #----------------------#
    
    
    
    ## ------ OverBought and OverSold Recency ---- ##
    list_1 = ['Stoch','RSI','TSI','UO','Williams %R','MFI']
    
    for i in list_1:
        df_prep[i+'_overbought_sum'] = 0
        df_prep[i+'_oversold_sum'] = 0
    
    
    for j in range(0,df_prep.shape[0]):
        for i in list_1:
#             df_prep[i+'_overbought_sum'] = 0
#             df_prep[i+'_oversold_sum'] = 0
        
            if j==0:
                df_prep[i+'_overbought_sum'][j] = df_prep[i+'_overbought'][j]
                
            elif(df_prep[i+'_overbought'][j]==1):
                df_prep[i+'_overbought_sum'][j] = df_prep[i+'_overbought'][j]+df_prep[i+'_overbought_sum'][j-1]
            else:
                df_prep[i+'_overbought_sum'][j] = 0
            
            
            if j==0:
                df_prep[i+'_oversold_sum'][j] = df_prep[i+'_oversold'][j]
                
            elif(df_prep[i+'_oversold'][j]==1):
                df_prep[i+'_oversold_sum'][j] = df_prep[i+'_oversold'][j]+df_prep[i+'_oversold_sum'][j-1]
            else:
                df_prep[i+'_oversold_sum'][j] = 0



    # ------ TTM Squeeze indicator ------#
    
    df_prep['30 Day MA'] = df_prep['askclose'].rolling(window = 20).mean()
    df_prep['30 Day STD'] = df_prep['askclose'].rolling(window = 20).std()
    df_prep['Upper Band'] = df_prep['30 Day MA'] + (df_prep['30 Day STD'] * 2)
    df_prep['Lower Band'] = df_prep['30 Day MA'] - (df_prep['30 Day STD'] * 2)

                
                
                
                
    

    print(" Data prep done ") 
    
    df_prep.ffill()
            
    final_data= pd.concat([final_data,df_prep],axis =0 , ignore_index =True)   
            
    clear_output()    
            
            
            

In [ ]:
# df_prep

In [ ]:
### creation of iteration table for data preparation
row_count.reset_index(inplace=True)
iteration_table = row_count[['Currency','TimeFrame']]

In [ ]:
##iteration_table = iteration_table.head(1)

In [ ]:

#iteration_table
global final_data
global counter 
counter = 0
final_data =pd.DataFrame()
data_prep = iteration_table.apply(lambda row :model_data_prep(row['Currency'],row['TimeFrame']),axis=1 )

In [ ]:
final_data.to_csv("<local-path>",index=False)